# Preprocessing GEE-Derived Landsat Vegetation Index GeoTIFFs

## Import libraries

In [1]:
import arcpy
import glob
import os

## Configurations

In [2]:
# GDB and GEE data paths
data_root_dir = r"C:\Users\Huiting Huang\Documents\GIS_Certificate_2025_26\ArcGIS_Projects\EvaluatingForestDisturbances"
gdb_path = os.path.join(data_root_dir, "EvaluatingForestDisturbances.gdb")
gee_landsat_dir = os.path.join(data_root_dir, "Data", "GEE_Landsat")

# Area of interest and processing extent
aoi = "North" # North, South
extent = f"TNC_AOIs_{aoi}"

# Spatial reference
input_gcs = arcpy.SpatialReference("GCS_WGS_1984")
output_coordinate_system = arcpy.SpatialReference("NAD 1983 UTM Zone 17N")
transformations = arcpy.ListTransformations(
    input_gcs,
    output_coordinate_system,
)
if not transformations:
    raise RuntimeError("No geographic transformation is available.")
geographic_transformation = transformations[0]

## Set geoprocessing environment

In [3]:
# Reset inherited ArcPy environment settings before processing
arcpy.env.workspace = None
arcpy.env.outputCoordinateSystem = None
arcpy.env.geographicTransformations = None
arcpy.env.cellSize = None
arcpy.env.snapRaster = None

# Set ArcPy environment settings for processing
arcpy.env.workspace = gdb_path
arcpy.env.overwriteOutput = True
#arcpy.env.addOutputsToMap = False
arcpy.env.outputCoordinateSystem = output_coordinate_system
arcpy.env.geographicTransformations = geographic_transformation
arcpy.env.extent = extent

## Import, convert and reproject GEE vegetation-index GeoTIFFs to raster datasets

In [ ]:
# Select parameters: the Landsat sensor, vegetation index, compositing method, and time period.

# Parameter options:
# landsat sensor: L5 (Landsat 5), L7 (Landsat 7), L8 (Landsat 8)
# time period: 06-08 (June–August), 08 (August), 07 (July), 06 (June)
# compositing method: Max, Mean, Median
# vegetation index: 
#   NDVI: normalized difference vegetation index, 
#   NDMI: normalized difference moisture index, 
#   EVI: enhanced vegetation index

dataset = "L5" # Options: L5, L7, L8
composite_strategy = "Max" # Options: Max, Mean, Median
vegetation_index = "EVI" # Options: NDVI, NDMI, EVI
time_period = "06-08" # Options: 06-08, 08, 07, 06

# Map each Landsat sensor to its input filename suffix and output name suffix.
suffixes_by_dataset = {
    "L5": ("_Landsat_WGS84", ""),
    "L7": ("_Landsat7_WGS84", "_L7"),
    "L8": ("_Landsat8_WGS84", "_L8"),
}
if dataset not in suffixes_by_dataset:
    raise ValueError(f"Unsupported Landsat dataset: {dataset}")

# Find matching GEE vegetation-index GeoTIFFs for the selected parameters.
input_suffix, output_suffix = suffixes_by_dataset[dataset]
tiff_pattern = (
    f"{composite_strategy}_{vegetation_index}*_{time_period}_"
    f"{aoi}{input_suffix}*.tif"
)
tiff_list = glob.glob(os.path.join(gee_landsat_dir, tiff_pattern))
if not tiff_list:
    raise FileNotFoundError(f"No GeoTIFFs matched the pattern: {tiff_pattern}")
print("Input GeoTIFFs:", tiff_list)

# Convert the GeoTIFFs to raster datasets in the GDB
created_raster_paths = []
for tiff_path in tiff_list:
    # Build the output raster name and pathfrom the input filename.
    input_filename = os.path.basename(tiff_path)
    input_raster_name = os.path.splitext(input_filename)[0]
    output_raster_name = (
        input_raster_name
        .replace("-", "To")
        .replace(input_suffix, output_suffix)
    )
    output_path = os.path.join(gdb_path, output_raster_name)

    print("Creating output raster:", output_raster_name)
    with arcpy.EnvManager(
        resamplingMethod="BILINEAR", 
        pyramid="PYRAMIDS -1 BILINEAR DEFAULT 75 NO_SKIP NO_SIPS"
    ):
        arcpy.management.CopyRaster(
            in_raster=tiff_path,
            out_rasterdataset=output_path,
            config_keyword="",
            background_value=None,
            nodata_value="nan",
            onebit_to_eightbit="NONE",
            colormap_to_RGB="NONE",
            pixel_type="",
            scale_pixel_value="NONE",
            RGB_to_Colormap="NONE",
            format="GRID",
            transform="NONE",
            process_as_multidimensional="CURRENT_SLICE",
            build_multidimensional_transpose="NO_TRANSPOSE"
        )
    print("Created output raster:", output_raster_name)
    created_raster_paths.append(output_path)

Input GeoTIFFs: ['C:\\Users\\Huiting Huang\\Documents\\GIS_Certificate_2025_26\\ArcGIS_Projects\\EvaluatingForestDisturbances\\Data\\GEE_Landsat\\Max_EVI_2002_06-08_North_Landsat_WGS84.tif', 'C:\\Users\\Huiting Huang\\Documents\\GIS_Certificate_2025_26\\ArcGIS_Projects\\EvaluatingForestDisturbances\\Data\\GEE_Landsat\\Max_EVI_2003_06-08_North_Landsat_WGS84.tif']
Creating output raster: Max_EVI_2002_06To08_North
Created output raster: Max_EVI_2002_06To08_North
Creating output raster: Max_EVI_2003_06To08_North
Created output raster: Max_EVI_2003_06To08_North


## Resample vegetation-index rasters to 30 m and snap to forest type raster

In [ ]:
# Map each AOI to its forest-type raster used for raster alignment.
snap_rasters_by_aoi = {
    "North": "fortype_aoiN_UTM_30m",
    "South": "fortype_aoiS_UTM_30m",
}

# Resample the created rasters to 30 m resolution and snap to the corresponding forest-type raster.
for input_raster_path in created_raster_paths:
    # Build the output raster name and path from the input filename.
    input_raster_name = os.path.basename(input_raster_path)
    output_raster_name = f"{input_raster_name}_30m"
    output_raster_path = os.path.join(gdb_path, output_raster_name)
    print("Creating resampled raster:", output_raster_name)
    with arcpy.EnvManager(
        resamplingMethod="BILINEAR",
        snapRaster=snap_rasters_by_aoi[aoi],
        pyramid="PYRAMIDS -1 BILINEAR DEFAULT 75 NO_SKIP NO_SIPS",
    ):
        arcpy.management.Resample(
            in_raster=input_raster_path,
            out_raster=output_raster_path,
            cell_size="30 30",
            resampling_type="BILINEAR",
        )
    print("Created resampled raster:", output_raster_name)

Creating resampled raster: Max_EVI_2002_06To08_North_30m
Created resampled raster: Max_EVI_2002_06To08_North_30m
Creating resampled raster: Max_EVI_2003_06To08_North_30m
Created resampled raster: Max_EVI_2003_06To08_North_30m
